# Day 4: จัดกลุ่มแหล่งที่มาของเหตุการณ์มลพิษ

เป้าหมายคือติดป้ายเหตุการณ์แต่ละอันจาก Day 3 ว่าเป็น `local-consistent`, `regional` หรือ `unclear`
โดยดู 3 สัญญาณร่วมกัน คือ ทิศลม, ค่าที่สถานีควบคุม และขนาดของ local increment

**หัวข้อ 4.1 ถึง 4.5 ยังไม่แตะข้อมูลมลพิษเลย** เป็นการนิยาม "ทิศนิคม" จากภูมิศาสตร์ล้วน ๆ ก่อน
เพราะแผนกำหนดว่าต้องเขียนกฎให้ชัดก่อนดูผล ไม่อย่างนั้นจะกลายเป็นการขยับเกณฑ์ไปเรื่อยจนได้คำตอบที่อยากได้

หมายเหตุเรื่องคำ: ใช้คำว่า "สอดคล้องกับ" ไม่ใช้ "เกิดจาก" เพราะลมทิศเดียวกันมีโรงงานหลายแห่ง
การวิเคราะห์นี้ชี้ความสอดคล้องเชิงสถิติเท่านั้น ไม่ใช่การพิสูจน์สาเหตุ

## 4.1 ขอบเขตพื้นที่อุตสาหกรรมจาก OpenStreetMap

ข้อมูลดึงด้วย `src/fetch_osm.py` ผ่าน Overpass API เอาทั้ง `landuse=industrial` และ `power=plant`
เพราะโรงไฟฟ้าบางแห่งไม่ได้ติดแท็ก landuse ไว้

ขอข้อมูลแบบ `out geom` คือเอาพิกัดทุกจุดของรูปหลายเหลี่ยม ไม่ใช่แค่จุดกึ่งกลาง
เหตุผลดูได้จากผลด้านล่าง คือ นิคมมาบตาพุดยาวถึง 9.6 กม. ถ้ายุบเหลือจุดเดียวจะคำนวณทิศผิดไปมาก

ข้อมูล OSM อยู่ภายใต้สัญญาอนุญาต ODbL ต้องให้เครดิตใน README

In [1]:
import json

import numpy as np
import pandas as pd
from matplotlib.path import Path as MplPath

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

# พิกัดสถานี (ชุดเดียวกับที่ใช้ดึงข้อมูลลมใน src/fetch_wind.py)
STATIONS = {
    225611: {'lat': 12.7087, 'lon': 101.1660, 'name': 'Health Promotion Hospital Maptaput', 'role': 'near'},
    7846:   {'lat': 12.7352, 'lon': 101.1356, 'name': 'Field Crop Research Center', 'role': 'near'},
    717:    {'lat': 12.9738, 'lon': 101.2129, 'name': 'Pluakdaeng District Health Office', 'role': 'control'},
}

osm = json.load(open('../data/raw/osm_industrial_sites.json', encoding='utf-8'))

polygons = []   # (ชื่อ, array ของ [lon, lat], ที่มาของรูปร่าง)
for el in osm['elements']:
    tags = el.get('tags', {})
    name = tags.get('name:en') or tags.get('name') or f"{el['type']}/{el['id']}"
    if el.get('geometry'):
        polygons.append((name, np.array([[p['lon'], p['lat']] for p in el['geometry']]), 'polygon'))
    elif el.get('bounds'):
        # relation ที่ Overpass ไม่ส่งรูปร่างสมาชิกมาด้วย ใช้กรอบสี่เหลี่ยมแทน
        b = el['bounds']
        polygons.append((name, np.array([
            [b['minlon'], b['minlat']], [b['maxlon'], b['minlat']],
            [b['maxlon'], b['maxlat']], [b['minlon'], b['maxlat']]]), 'bbox'))

n_bbox = sum(1 for _, _, src in polygons if src == 'bbox')
print(f'พื้นที่อุตสาหกรรมจาก OSM: {len(polygons)} แห่ง (ใช้กรอบสี่เหลี่ยมแทนรูปร่างจริง {n_bbox} แห่ง)')
print()
print('นิคม/โรงไฟฟ้าที่มีชื่อ เรียงตามความกว้าง 8 อันดับแรก:')
rows = []
for name, ring, src in polygons:
    if name.startswith(('way/', 'relation/')):
        continue
    lat0 = ring[:, 1].mean()
    rows.append({
        'ชื่อ': name[:40],
        'lat': round(lat0, 4), 'lon': round(ring[:, 0].mean(), 4),
        'กว้างกม.': round((ring[:, 0].max() - ring[:, 0].min()) * 111.32 * np.cos(np.radians(lat0)), 1),
        'สูงกม.': round((ring[:, 1].max() - ring[:, 1].min()) * 110.54, 1),
    })
print(pd.DataFrame(rows).sort_values('กว้างกม.', ascending=False).head(8).to_string(index=False))

พื้นที่อุตสาหกรรมจาก OSM: 277 แห่ง (ใช้กรอบสี่เหลี่ยมแทนรูปร่างจริง 4 แห่ง)

นิคม/โรงไฟฟ้าที่มีชื่อ เรียงตามความกว้าง 8 อันดับแรก:
                                    ชื่อ     lat      lon  กว้างกม.  สูงกม.
Eastern Seaboard Industrial Estate Rayon 13.0097 101.1781       6.3     6.2
           Map Ta Phut Industrial Estate 12.6767 101.1493       5.7     9.6
     Amata City Rayong Industrial Estate 12.9722 101.1097       4.9     8.5
WHA Eastern Seaboard Industrial Estate 1 13.0615 101.1942       4.4     5.0
WHA Eastern Industrial Estate (Map Ta Ph 12.7099 101.1182       3.5     3.2
WHA Eastern Seaboard Industrial Estate 3 13.1018 101.2729       3.4     2.0
                  CPGC Industrial Estate 12.7982 101.2087       3.3     3.6
              WHA Rayong Industrial Land 12.8276 101.2522       3.1     2.7


**อ่านผล:** รอบมาบตาพุดไม่ได้มีนิคมเดียว แต่เป็นแนวนิคมยาวตลอดชายฝั่งระยอง
นิคมที่ใหญ่ที่สุดหลายแห่งอยู่ทางเหนือ (Eastern Seaboard, Amata City, WHA) ซึ่งใกล้สถานีควบคุม 717 มากกว่าสถานีใกล้นิคมเสียอีก
ประเด็นนี้จะกลับมาสำคัญในหัวข้อ 4.5

## 4.2 แปลงพื้นที่อุตสาหกรรมเป็นตารางกริด

ถ้านับนิคมละ 1 จุด นิคมมาบตาพุดที่กว้าง 5.7 คูณ 9.6 กม. จะถูกนับเท่ากับโรงงานเล็ก 1 โรง
เลยแบ่งพื้นที่เป็นช่องตารางละ 200 เมตร แล้วเช็กว่าช่องไหนตกอยู่ในรูปหลายเหลี่ยมบ้าง
ใช้ `matplotlib.path` ทำ point-in-polygon เพราะมีอยู่ใน requirements แล้ว ไม่ต้องลงไลบรารี GIS เพิ่ม

In [2]:
# แปลงพื้นที่อุตสาหกรรมเป็นตารางกริด 200 เมตร เพื่อให้ถ่วงน้ำหนักตามพื้นที่จริงได้
# (ถ้านับนิคมละ 1 จุด นิคมมาบตาพุดที่ยาว 9.6 กม. จะถูกนับเท่ากับโรงงานเล็ก 1 โรง)
GRID_M = 200
LAT0 = 12.85                                        # ละติจูดอ้างอิง ใช้แปลงองศาเป็นเมตร
M_PER_DEG_LAT = 110540.0
M_PER_DEG_LON = 111320.0 * np.cos(np.radians(LAT0))

all_lon = np.concatenate([r[:, 0] for _, r, _ in polygons])
all_lat = np.concatenate([r[:, 1] for _, r, _ in polygons])
LON, LAT = np.meshgrid(np.arange(all_lon.min(), all_lon.max(), GRID_M / M_PER_DEG_LON),
                       np.arange(all_lat.min(), all_lat.max(), GRID_M / M_PER_DEG_LAT))
pts = np.column_stack([LON.ravel(), LAT.ravel()])

is_industrial = np.zeros(len(pts), dtype=bool)
for name, ring, src in polygons:
    lo, hi = ring.min(axis=0), ring.max(axis=0)
    box = ((pts[:, 0] >= lo[0]) & (pts[:, 0] <= hi[0]) &
           (pts[:, 1] >= lo[1]) & (pts[:, 1] <= hi[1]) & ~is_industrial)
    if not box.any():
        continue
    is_industrial[np.where(box)[0][MplPath(ring).contains_points(pts[box])]] = True

ind_pts = pts[is_industrial]
print(f'กริดทั้งหมด {len(pts):,} ช่อง')
print(f'ช่องที่อยู่ในพื้นที่อุตสาหกรรม {len(ind_pts):,} ช่อง = {len(ind_pts) * (GRID_M/1000)**2:.1f} ตร.กม.')

กริดทั้งหมด 63,332 ช่อง
ช่องที่อยู่ในพื้นที่อุตสาหกรรม 4,204 ช่อง = 168.2 ตร.กม.


## 4.3 ทิศของพื้นที่อุตสาหกรรมเมื่อมองจากแต่ละสถานี

แบ่งเป็น 16 ทิศ ทิศละ 22.5 องศา แล้วคิด 2 แบบ
- `%พื้นที่` คือสัดส่วนพื้นที่อุตสาหกรรมดิบ ๆ ในทิศนั้น
- `%ถ่วงระยะ` คือถ่วงน้ำหนักด้วย 1 ส่วนระยะทางยกกำลังสอง เพราะโรงงานห่าง 1 กม. กับห่าง 9 กม. ส่งผลต่อสถานีไม่เท่ากัน

ตัว 1/d² เป็นตัวแทนคร่าว ๆ ของการเจือจางตามระยะทาง ไม่ใช่แบบจำลองการกระจายตัวจริง
ที่ต้องใส่พื้นระยะต่ำสุด 200 เมตรไว้ เพราะสถานี 225611 อยู่ติดพื้นที่อุตสาหกรรมมาก ถ้าไม่ใส่ค่าจะระเบิด

In [3]:
# ทิศ (bearing) วัดแบบเข็มทิศ 0=เหนือ 90=ออก 180=ใต้ 270=ตก
# ทิศลมในอุตุนิยมวิทยาคือทิศที่ลม "พัดมาจาก" ดังนั้นถ้านิคมอยู่ทางทิศ 180 จากสถานี
# ลมที่พามลพิษจากนิคมเข้าสถานีก็คือลมทิศ 180 เหมือนกัน เทียบกันตรง ๆ ได้เลย
RADIUS_KM = 10        # ไกลกว่านี้ถือว่าผลน้อย และแนวคิด local increment ใช้ไม่ได้แล้ว
MIN_DIST_M = 200      # พื้นระยะ กันหารศูนย์ กรณีสถานีตั้งอยู่ในพื้นที่อุตสาหกรรมพอดี
LABELS = ['N','NNE','NE','ENE','E','ESE','SE','SSE','S','SSW','SW','WSW','W','WNW','NW','NNW']


def station_sectors(lat, lon):
    """คืนค่า (เลขทิศ 0-15, น้ำหนักถ่วงระยะ, ระยะเป็นเมตร) ของช่องกริดอุตสาหกรรมรอบจุดหนึ่ง"""
    dx = (ind_pts[:, 0] - lon) * M_PER_DEG_LON       # ตะวันออกเป็นบวก
    dy = (ind_pts[:, 1] - lat) * M_PER_DEG_LAT       # เหนือเป็นบวก
    dist = np.sqrt(dx**2 + dy**2)
    keep = dist <= RADIUS_KM * 1000
    dx, dy, dist = dx[keep], dy[keep], np.maximum(dist[keep], MIN_DIST_M)
    bearing = np.degrees(np.arctan2(dx, dy)) % 360
    sector = np.floor(((bearing + 11.25) % 360) / 22.5).astype(int)
    return sector, 1.0 / dist**2, dist              # น้ำหนัก 1/d^2 แทนการเจือจางตามระยะ


sector_table = {}
for sid, s in STATIONS.items():
    sector, weight, dist = station_sectors(s['lat'], s['lon'])
    area = np.bincount(sector, minlength=16).astype(float)
    wsum = np.bincount(sector, weights=weight, minlength=16)
    sector_table[sid] = pd.DataFrame({
        'ทิศ': LABELS,
        'องศา': [f'{(i*22.5-11.25) % 360:.0f}-{(i*22.5+11.25) % 360:.0f}' for i in range(16)],
        '%พื้นที่': (100 * area / area.sum()).round(1),
        '%ถ่วงระยะ': (100 * wsum / wsum.sum()).round(1),
    })
    print(f"=== {sid} {s['name']} ({s['role']}) ===")
    print(f'พื้นที่อุตสาหกรรมในรัศมี {RADIUS_KM} กม. = {area.sum() * 0.04:.1f} ตร.กม. '
          f'| ใกล้สุด {dist.min()/1000:.2f} กม.')
    print(sector_table[sid].sort_values('%ถ่วงระยะ', ascending=False).head(6).to_string(index=False))
    print()

=== 225611 Health Promotion Hospital Maptaput (near) ===
พื้นที่อุตสาหกรรมในรัศมี 10 กม. = 39.7 ตร.กม. | ใกล้สุด 0.76 กม.
ทิศ    องศา  %พื้นที่  %ถ่วงระยะ
WSW 236-259      18.6       23.1
  W 259-281      25.8       19.6
 SW 214-236      14.5       19.1
SSW 191-214       9.8       17.7
WNW 281-304       3.8        8.2
  N  349-11       9.0        4.0

=== 7846 Field Crop Research Center (near) ===
พื้นที่อุตสาหกรรมในรัศมี 10 กม. = 40.7 ตร.กม. | ใกล้สุด 1.43 กม.
ทิศ    องศา  %พื้นที่  %ถ่วงระยะ
SSE 146-169      24.0       22.0
SSW 191-214      13.4       13.8
  S 169-191      18.6       13.7
 SE 124-146       4.2       10.1
 SW 214-236      10.9        8.7
ENE   56-79       6.2        8.2

=== 717 Pluakdaeng District Health Office (control) ===
พื้นที่อุตสาหกรรมในรัศมี 10 กม. = 40.9 ตร.กม. | ใกล้สุด 2.49 กม.
ทิศ    องศา  %พื้นที่  %ถ่วงระยะ
 NW 304-326      22.4       35.7
NNW 326-349      29.7       31.3
WNW 281-304      14.4       13.3
  N  349-11      14.0        7.9
  W 259-281     

**อ่านผล:**
- สถานี 225611 มีนิคมอยู่ทางทิศตะวันตกเฉียงใต้ถึงตะวันตก และใกล้มากคือ 0.76 กม.
- สถานี 7846 มีนิคมอยู่ทางทิศใต้ค่อนไปทางตะวันออกเฉียงใต้ ซึ่งคนละทิศกับ 225611 เพราะตัวสถานีอยู่เหนือขึ้นไป
- สถานีควบคุม 717 มีพื้นที่อุตสาหกรรมในรัศมี 10 กม. ถึง 40.9 ตร.กม. ซึ่ง**มากกว่าสถานีใกล้นิคมทั้งสองแห่ง**

ข้อสุดท้ายขัดกับสมมติฐานที่ใช้มาตั้งแต่ Day 1 ว่า 717 เป็นสถานีพื้นหลังที่สะอาด จึงต้องตรวจต่อในหัวข้อ 4.5

## 4.4 นิยามช่วงองศา "ทิศนิคม" ของแต่ละสถานี

กฎที่ตั้งไว้ก่อนดูข้อมูลมลพิษ คือเรียงทิศจากน้ำหนักมากไปน้อย แล้วเลือกมาจนรวมได้ถึง 80%

In [4]:
# กฎที่ตั้งไว้ "ก่อน" ดูข้อมูลมลพิษ: ทิศนิคม = กลุ่มทิศที่รวมน้ำหนักถ่วงระยะได้ถึง 80%
THRESHOLD = 0.80
industrial_sectors = {}
for sid, s in STATIONS.items():
    t = sector_table[sid].sort_values('%ถ่วงระยะ', ascending=False)
    chosen = t[t['%ถ่วงระยะ'].cumsum().shift(1, fill_value=0) < THRESHOLD * 100]
    industrial_sectors[sid] = sorted(chosen['ทิศ'], key=LABELS.index)
    print(f"{sid} ({s['role']}): {' '.join(industrial_sectors[sid])} "
          f"= {chosen['%ถ่วงระยะ'].sum():.0f}% ของน้ำหนักถ่วงระยะ")

225611 (near): SSW SW WSW W WNW = 88% ของน้ำหนักถ่วงระยะ
7846 (near): NE ENE SE SSE S SSW SW = 83% ของน้ำหนักถ่วงระยะ
717 (control): WNW NW NNW = 80% ของน้ำหนักถ่วงระยะ


## 4.5 ตรวจสมมติฐาน: สถานีควบคุม 717 สะอาดจริงหรือไม่

เทียบพื้นที่อุตสาหกรรมรอบสถานี OpenAQ ทุกแห่งที่สำรวจไว้ตอน Day 1 ด้วยเกณฑ์เดียวกัน

In [5]:
# ตรวจสมมติฐานสำคัญ: สถานีควบคุม 717 "สะอาด" จริงหรือไม่
# เทียบกับสถานี OpenAQ ทุกแห่งที่สำรวจไว้ตอน Day 1
locs = json.load(open('../data/raw/openaq_locations.json', encoding='utf-8'))
cand = {int(l['id']): {'lat': l['coordinates']['latitude'], 'lon': l['coordinates']['longitude'],
                       'name': l['name']} for l in locs}
cand[717] = STATIONS[717]

rows = []
for sid, s in sorted(cand.items()):
    dx = (ind_pts[:, 0] - s['lon']) * M_PER_DEG_LON
    dy = (ind_pts[:, 1] - s['lat']) * M_PER_DEG_LAT
    d = np.sqrt(dx**2 + dy**2) / 1000
    d_main = np.sqrt((((s['lon'] - STATIONS[225611]['lon']) * M_PER_DEG_LON)**2 +
                      ((s['lat'] - STATIONS[225611]['lat']) * M_PER_DEG_LAT)**2)) / 1000
    rows.append({'station_id': sid, 'name': s['name'][:34],
                 'ห่างจาก225611กม.': round(d_main, 1),
                 'อุตฯใกล้สุดกม.': round(d.min(), 2),
                 'ตร.กม.ใน5กม.': round((d <= 5).sum() * 0.04, 1),
                 'ตร.กม.ใน10กม.': round((d <= 10).sum() * 0.04, 1)})
print('พื้นที่อุตสาหกรรมรอบสถานีแต่ละแห่ง (เรียงจากสะอาดสุด):')
print(pd.DataFrame(rows).sort_values('ตร.กม.ใน10กม.').to_string(index=False))

พื้นที่อุตสาหกรรมรอบสถานีแต่ละแห่ง (เรียงจากสะอาดสุด):
 station_id                               name  ห่างจาก225611กม.  อุตฯใกล้สุดกม.  ตร.กม.ใน5กม.  ตร.กม.ใน10กม.
       7676                   Tha Pradu, Mueng              12.1            3.35           2.5            5.3
     225576                 Khai Surasinghanat              18.2            0.39           5.3            5.3
     225690                Agricultural Office              12.6            2.53           3.7            5.3
    2912858                    โรงเรียนวัดเกาะ              13.0            6.49           0.0           10.1
    3345345          Rayong Provincial Stadium               8.1            6.98           0.0           16.0
    2847196                  โรงเรียนวัดกระเฉท               8.2            2.56           3.2           21.8
        863                  Noen Phra, Mueang               1.8            2.03          12.4           39.6
       6023                Map Ta Phut, Mueang               0.0 

**อ่านผล (ข้อค้นพบสำคัญของวันนี้):**

สถานีควบคุม 717 Pluakdaeng มีพื้นที่อุตสาหกรรมในรัศมี 10 กม. มากที่สุดในบรรดาสถานีทั้งหมด คือ 40.9 ตร.กม.
สาเหตุคือมันอยู่กลางวงล้อมของนิคม Eastern Seaboard, Amata City Rayong และ WHA ซึ่งอยู่ทางทิศ WNW ถึง NNW ของมัน

ข้อดีที่ยังเหลืออยู่คือในรัศมี 5 กม. มีแค่ 5.2 ตร.กม. (เทียบกับ 20.5 ของสถานี 225611) แปลว่าไม่ได้ติดนิคมแบบสถานีใกล้นิคม
แต่ก็ไม่ใช่สถานีพื้นหลังที่สะอาดอย่างที่สมมติไว้ตอนแรก

**ผลที่ตามมา:** `local increment` ที่คำนวณไว้ตอน Day 3 ไม่ใช่ "ค่าในเขตอุตสาหกรรม ลบ ค่าพื้นที่สะอาด"
แต่เป็น "ค่าในเขตอุตสาหกรรมหนึ่ง ลบ ค่าในเขตอุตสาหกรรมอีกแห่ง" ซึ่งทำให้ increment ต่ำกว่าความจริงเวลาลมพัดนิคมเข้าสถานีควบคุมพอดี

**วิธีรับมือที่ใช้ต่อจากนี้** ตรงกับที่แผนเตือนไว้ว่าถ้าลมพัดจากนิคมไปหาสถานีพื้นหลังให้ตัดชั่วโมงแบบนั้นออก
1. ตัดชั่วโมงที่ลมที่สถานี 717 พัดมาจากทิศ WNW NW NNW ออกจากการเทียบ increment
2. เขียนข้อจำกัดนี้ลง README ตรง ๆ พร้อมตัวเลข 40.9 ตร.กม.

**ทางเลือกที่ดีกว่าแต่ต้องใช้เวลาเพิ่ม:** สถานี 225690 Agricultural Office มีพื้นที่อุตสาหกรรมในรัศมี 10 กม. แค่ 5.3 ตร.กม.
ห่างจากสถานีหลัก 12.6 กม. และมีข้อมูลครบทั้ง SO2 NO2 PM2.5 ช่วงปี 2021 ถึง 2025 ซึ่งทับกับช่วงข้อมูลของสถานี 225611
ถ้าเวลาพอควรดึงมาเป็นสถานีควบคุมที่สอง แล้วดูว่าผลการติดป้ายเปลี่ยนไปมากไหม

## 4.6 เตรียมข้อมูลรายชั่วโมงสำหรับการติดป้าย

In [6]:
# เตรียมข้อมูลรายชั่วโมง เลือกแหล่งลม และดึงสถานะของสถานีควบคุมมาเทียบ
POLLUTANTS = ['so2', 'no2', 'pm25', 'pm10', 'o3', 'co']


def to_sector(deg):
    """แปลงองศาเป็นเลขทิศ 0-15 (ค่าว่างคืน -1)"""
    return np.floor(((deg + 11.25) % 360) / 22.5).fillna(-1).astype(int)


near = pd.read_parquet('../data/processed/hourly_with_increment.parquet')
flagged = pd.read_parquet('../data/processed/hourly_flagged.parquet')

# ลมหลักใช้ IFS (grid 9 กม. แยกสถานีได้จริง) ชั่วโมงไหนไม่มีค่อยถอยไปใช้ ERA5 (grid 25 กม.)
for df in (near, flagged):
    df['wind_dir'] = df['ifs_wind_direction_10m'].fillna(df['era5_wind_direction_10m'])
    df['wind_src'] = np.where(df['ifs_wind_direction_10m'].notna(), 'ifs',
                              np.where(df['era5_wind_direction_10m'].notna(), 'era5', 'none'))
near['sector'] = to_sector(near['wind_dir'])

# สถานะสถานีควบคุม 717 รายชั่วโมง: ผิดปกติไหม และลมพัดมาจากนิคมของมันเองไหม
ctrl = flagged[flagged.station_id == 717].copy()
ctrl['ctrl_contaminated'] = to_sector(ctrl['wind_dir']).map(
    lambda s: LABELS[s] in industrial_sectors[717] if s >= 0 else False)
ctrl = ctrl[['datetime_utc', 'ctrl_contaminated'] + [f'{p}_episode_flag' for p in POLLUTANTS]]
ctrl = ctrl.rename(columns={f'{p}_episode_flag': f'{p}_ctrl_flag' for p in POLLUTANTS})

hourly = near.merge(ctrl, on='datetime_utc', how='left')
print(f'ข้อมูลรายชั่วโมงของสถานีใกล้นิคม {len(hourly):,} แถว')
print('แหล่งลมที่ได้ใช้จริง:')
print(hourly.groupby(['station_id', 'wind_src']).size().unstack(fill_value=0).to_string())
print()
print(f"ชั่วโมงที่ลมพัดจากนิคมเข้าสถานีควบคุมเอง (ตัดออกตอนเทียบ increment): "
      f"{hourly['ctrl_contaminated'].sum():,} จาก {hourly['ctrl_contaminated'].notna().sum():,} ชั่วโมง")

ข้อมูลรายชั่วโมงของสถานีใกล้นิคม 66,852 แถว
แหล่งลมที่ได้ใช้จริง:
wind_src    era5    ifs
station_id             
7846        2336  34755
225611         0  29761

ชั่วโมงที่ลมพัดจากนิคมเข้าสถานีควบคุมเอง (ตัดออกตอนเทียบ increment): 3,173 จาก 62,679 ชั่วโมง


## 4.7 เกณฑ์ local increment

ต้องมีเกณฑ์ว่า increment "สูงพอ" แค่ไหนถึงนับว่าสอดคล้องกับแหล่งในพื้นที่
เลือกใช้เปอร์เซ็นไทล์ที่ 75 ของ increment ในชั่วโมงปกติของสารนั้นที่สถานีนั้น
ข้อดีคือเกณฑ์มาจากข้อมูลเอง ไม่ได้ตั้งลอย ๆ และปรับตามแต่ละสารกับแต่ละสถานีโดยอัตโนมัติ

In [7]:
# เกณฑ์ increment ตั้งไว้ล่วงหน้า ต้องสูงกว่าเปอร์เซ็นไทล์ที่ 75 ของ increment ปกติของสารนั้น
# นับเฉพาะชั่วโมงที่สถานีควบคุมไม่ได้โดนลมจากนิคมของตัวเอง
clean = hourly[hourly['ctrl_contaminated'] == False]
INC_P75 = {}
for p in POLLUTANTS:
    for sid in (225611, 7846):
        v = clean.loc[clean.station_id == sid, f'{p}_increment'].dropna()
        INC_P75[(sid, p)] = float(np.percentile(v, 75)) if len(v) >= 100 else np.nan
print('เกณฑ์ increment (p75 ของชั่วโมงที่สถานีควบคุมไม่ปนเปื้อน):')
print(pd.DataFrame([{'station_id': k[0], 'สาร': k[1], 'p75': v} for k, v in INC_P75.items()])
      .pivot(index='สาร', columns='station_id', values='p75').round(4).to_string())
print()
print('หมายเหตุ: CO ไม่มีเกณฑ์เพราะสถานีควบคุมไม่มีข้อมูล CO เหตุการณ์ CO จึงติดป้าย unclear-no-data ทั้งหมด')

เกณฑ์ increment (p75 ของชั่วโมงที่สถานีควบคุมไม่ปนเปื้อน):
station_id  7846    225611
สาร                       
co             NaN     NaN
no2          0.006   0.010
o3           0.015   0.011
pm10        -1.000  16.000
pm25         3.500   6.000
so2          0.002   0.003

หมายเหตุ: CO ไม่มีเกณฑ์เพราะสถานีควบคุมไม่มีข้อมูล CO เหตุการณ์ CO จึงติดป้าย unclear-no-data ทั้งหมด


## 4.8 สร้างตัวแปรระดับเหตุการณ์

แต่ละเหตุการณ์สรุปออกมาเป็น 4 ตัวเลขหลัก
- `frac_wind_from_industry` สัดส่วนชั่วโมงที่ลมพัดมาจากทิศนิคม
- `frac_ctrl_flagged` สัดส่วนชั่วโมงที่สถานีควบคุมผิดปกติด้วยในสารเดียวกัน
- `median_increment` ค่ากลางของ increment เฉพาะชั่วโมงที่สถานีควบคุมไม่ปนเปื้อน
- `wind_coverage` กับ `ctrl_coverage` ไว้ดูว่าข้อมูลพอให้ตัดสินไหม

In [8]:
# สร้างตัวแปรระดับเหตุการณ์ ติดป้ายเฉพาะเหตุการณ์ที่สถานีใกล้นิคม
episodes = pd.read_csv('../data/processed/episodes.csv', parse_dates=['start_utc', 'end_utc'])
episodes = episodes[episodes.role == 'near'].reset_index(drop=True)
by_station = {sid: g.set_index('datetime_utc').sort_index() for sid, g in hourly.groupby('station_id')}

rows = []
for ep in episodes.itertuples(index=False):
    g = by_station[ep.station_id].loc[ep.start_utc:ep.end_utc]
    n = len(g)
    has_wind = g['sector'] >= 0
    in_ind = g.loc[has_wind, 'sector'].map(lambda s: LABELS[s] in industrial_sectors[ep.station_id])
    ctrl_flag = g[f'{ep.pollutant}_ctrl_flag']
    inc = g.loc[g['ctrl_contaminated'] == False, f'{ep.pollutant}_increment'].dropna()
    rows.append({
        'station_id': ep.station_id, 'pollutant': ep.pollutant,
        'start_utc': ep.start_utc, 'end_utc': ep.end_utc,
        'duration_hours': ep.duration_hours, 'peak_value': ep.peak_value, 'peak_z': ep.peak_z,
        'n_hours': n,
        'wind_coverage': has_wind.sum() / n if n else 0.0,
        'frac_wind_from_industry': in_ind.mean() if has_wind.any() else np.nan,
        'ctrl_coverage': ctrl_flag.notna().sum() / n if n else 0.0,
        'frac_ctrl_flagged': ctrl_flag.fillna(False).mean() if n else np.nan,
        'n_hours_increment': len(inc),
        'median_increment': inc.median() if len(inc) else np.nan,
        'inc_threshold': INC_P75[(ep.station_id, ep.pollutant)],
    })
feat = pd.DataFrame(rows)
print(f'สร้างตัวแปรให้เหตุการณ์ {len(feat):,} อัน')
print()
print('ความครบของข้อมูลที่ใช้ตัดสิน:')
print(f"  มีลมเกินครึ่งช่วงเหตุการณ์:        {(feat.wind_coverage > 0.5).sum():,} อัน ({100*(feat.wind_coverage > 0.5).mean():.1f}%)")
print(f"  มีข้อมูลสถานีควบคุมเกินครึ่ง:      {(feat.ctrl_coverage > 0.5).sum():,} อัน ({100*(feat.ctrl_coverage > 0.5).mean():.1f}%)")
print(f"  มี increment ใช้ได้อย่างน้อย 1 ชม.: {(feat.n_hours_increment > 0).sum():,} อัน ({100*(feat.n_hours_increment > 0).mean():.1f}%)")
print()
print(feat.head(5).to_string(index=False))

สร้างตัวแปรให้เหตุการณ์ 1,796 อัน

ความครบของข้อมูลที่ใช้ตัดสิน:
  มีลมเกินครึ่งช่วงเหตุการณ์:        1,796 อัน (100.0%)
  มีข้อมูลสถานีควบคุมเกินครึ่ง:      1,666 อัน (92.8%)
  มี increment ใช้ได้อย่างน้อย 1 ชม.: 1,430 อัน (79.6%)

 station_id pollutant                 start_utc                   end_utc  duration_hours  peak_value  peak_z  n_hours  wind_coverage  frac_wind_from_industry  ctrl_coverage  frac_ctrl_flagged  n_hours_increment  median_increment  inc_threshold
       7846       so2 2016-04-08 08:00:00+00:00 2016-04-08 08:00:00+00:00               1       0.011  6.7450        1            1.0                      1.0       1.000000                0.0                  1             0.009          0.002
       7846       so2 2016-04-10 08:00:00+00:00 2016-04-10 10:00:00+00:00               3       0.018  6.0705        3            1.0                      1.0       0.666667                0.0                  2             0.013          0.002
       7846       so2 2016-04-11

## 4.9 กฎการติดป้าย

เขียนกฎไว้ก่อนดูผล ไล่ตามลำดับ ข้อแรกที่เข้าเงื่อนไขเป็นผู้ชนะ

1. `unclear-no-data` ข้อมูลไม่พอตัดสิน คือมีลมหรือข้อมูลสถานีควบคุมไม่ถึงครึ่งของช่วงเหตุการณ์ หรือไม่มี increment ให้ใช้เลย
2. `regional` สถานีควบคุมผิดปกติในสารเดียวกันตั้งแต่ครึ่งหนึ่งของชั่วโมงขึ้นไป แปลว่าค่าสูงพร้อมกันทั้งสองที่
3. `local-consistent` ลมพัดจากทิศนิคมอย่างน้อย 60% ของชั่วโมง **และ** increment สูงกว่าเกณฑ์ p75
4. `unclear-mixed` ไม่เข้าข้อไหนเลย

In [9]:
# กฎติดป้าย เขียนไว้ก่อนดูผล ไล่ตามลำดับ ข้อแรกที่เข้าเงื่อนไขชนะ
WIND_MIN = 0.60     # ลมต้องพัดมาจากทิศนิคมอย่างน้อย 60% ของชั่วโมงในเหตุการณ์
CTRL_MIN = 0.50     # ถ้าสถานีควบคุมผิดปกติด้วยตั้งแต่ครึ่งหนึ่งของชั่วโมงขึ้นไป ถือว่าเป็นระดับภูมิภาค


def label_episodes(f, wind_min=WIND_MIN, ctrl_min=CTRL_MIN, inc_col='inc_threshold'):
    not_enough = ((f.wind_coverage <= 0.5) | (f.ctrl_coverage <= 0.5) |
                  (f.n_hours_increment == 0) | f[inc_col].isna())
    return np.select([not_enough, f.frac_ctrl_flagged >= ctrl_min,
                      (f.frac_wind_from_industry >= wind_min) & (f.median_increment > f[inc_col])],
                     ['unclear-no-data', 'regional', 'local-consistent'], default='unclear-mixed')


feat['label'] = label_episodes(feat)
print('ผลการติดป้ายทั้งหมด:')
print(feat['label'].value_counts().to_string())
print()
print('แยกตามสถานีและสาร (%):')
tab = pd.crosstab([feat.station_id, feat.pollutant], feat.label, normalize='index').mul(100).round(1)
print(tab.join(pd.crosstab([feat.station_id, feat.pollutant], feat.label).sum(axis=1).rename('จำนวน')).to_string())

ผลการติดป้ายทั้งหมด:
label
local-consistent    954
unclear-no-data     393
unclear-mixed       360
regional             89

แยกตามสถานีและสาร (%):
                      local-consistent  regional  unclear-mixed  unclear-no-data  จำนวน
station_id pollutant                                                                   
7846       co                      0.0       0.0            0.0            100.0     63
           no2                    50.5       3.3           25.3             20.9     91
           o3                     60.4       2.2           16.5             20.9    182
           pm10                   11.1      33.3           44.4             11.1      9
           pm25                   19.7      41.0           34.4              4.9     61
           so2                    63.5       2.1           11.9             22.5    427
225611     co                      0.0       0.0            0.0            100.0      3
           no2                    38.7       0.0           54

**อ่านผล:** รูปแบบที่ออกมาตรงกับความรู้เชิงกายภาพอย่างชัดเจน
- SO2 ได้ `local-consistent` ประมาณ 60 ถึง 64% และแทบไม่มี `regional` เลย ซึ่งสอดคล้องกับก๊าซจากการเผาไหม้ที่มีแหล่งอยู่ใกล้ ๆ
- PM2.5 กลับกัน คือได้ `regional` 41 ถึง 47% และ `local-consistent` แค่ 10 ถึง 20% ซึ่งสอดคล้องกับฝุ่นที่พัดมาเป็นวงกว้าง
- CO ติด `unclear-no-data` ทั้งหมด เพราะสถานีควบคุมไม่มีข้อมูล CO ให้เทียบ
- PM10 ที่สถานี 225611 ติด `unclear-no-data` ถึง 91.7% เพราะข้อมูล PM10 ของคู่สถานีทับกันน้อย

## 4.10 ทดสอบความไวต่อเกณฑ์

ป้ายกลุ่มไม่มีคำตอบจริงให้เทียบ สิ่งที่ทำได้คือรายงานว่าผลนิ่งแค่ไหนเมื่อขยับเกณฑ์

In [10]:
# ทดสอบความไวต่อเกณฑ์ เพราะไม่มีคำตอบจริงให้เทียบ ต้องรู้ว่าผลนิ่งแค่ไหน
base = feat['label'].copy()
rows = []
for wmin in (0.5, 0.6, 0.7):
    for cmin in (0.3, 0.5, 0.7):
        lab = label_episodes(feat, wind_min=wmin, ctrl_min=cmin)
        rows.append({'เกณฑ์ลม': wmin, 'เกณฑ์ควบคุม': cmin,
                     'local %': round(100 * (lab == 'local-consistent').mean(), 1),
                     'regional %': round(100 * (lab == 'regional').mean(), 1),
                     'unclear %': round(100 * np.isin(lab, ['unclear-mixed', 'unclear-no-data']).mean(), 1),
                     'ป้ายที่เปลี่ยน %': round(100 * (lab != base).mean(), 1)})
print('ความไวต่อเกณฑ์ลมและเกณฑ์สถานีควบคุม:')
print(pd.DataFrame(rows).to_string(index=False))
print()

rows = []
for q in (50, 75, 90):
    f2 = feat.copy()
    f2['thr_q'] = [float(np.percentile(clean.loc[clean.station_id == r.station_id,
                                                 f'{r.pollutant}_increment'].dropna(), q))
                   if len(clean.loc[clean.station_id == r.station_id, f'{r.pollutant}_increment'].dropna()) >= 100
                   else np.nan for r in f2.itertuples(index=False)]
    lab = label_episodes(f2, inc_col='thr_q')
    rows.append({'เกณฑ์ increment': f'p{q}',
                 'local %': round(100 * (lab == 'local-consistent').mean(), 1),
                 'regional %': round(100 * (lab == 'regional').mean(), 1),
                 'unclear %': round(100 * np.isin(lab, ['unclear-mixed', 'unclear-no-data']).mean(), 1),
                 'ป้ายที่เปลี่ยน %': round(100 * (lab != base).mean(), 1)})
print('ความไวต่อเกณฑ์ increment:')
print(pd.DataFrame(rows).to_string(index=False))

ความไวต่อเกณฑ์ลมและเกณฑ์สถานีควบคุม:
 เกณฑ์ลม  เกณฑ์ควบคุม  local %  regional %  unclear %  ป้ายที่เปลี่ยน %
     0.5          0.3     55.1         5.4       39.5               2.6
     0.5          0.5     55.2         5.0       39.8               2.1
     0.5          0.7     55.5         4.2       40.4               2.9
     0.6          0.3     53.0         5.4       41.6               0.4
     0.6          0.5     53.1         5.0       41.9               0.0
     0.6          0.7     53.3         4.2       42.5               0.8
     0.7          0.3     51.9         5.4       42.7               1.4
     0.7          0.5     52.1         5.0       43.0               1.1
     0.7          0.7     52.2         4.2       43.7               1.8

ความไวต่อเกณฑ์ increment:
เกณฑ์ increment  local %  regional %  unclear %  ป้ายที่เปลี่ยน %
            p50     53.5         5.0       41.5               0.4
            p75     53.1         5.0       41.9               0.0
            p90   

**อ่านผล:** ขยับเกณฑ์ลมในช่วง 0.5 ถึง 0.7 และเกณฑ์สถานีควบคุมในช่วง 0.3 ถึง 0.7
ป้ายเปลี่ยนไม่เกิน 2.9% ของเหตุการณ์ทั้งหมด ส่วนเกณฑ์ increment ที่ p50 เปลี่ยนแค่ 0.4%
มีแค่ p90 ที่เปลี่ยนถึง 6.0% ซึ่งยังถือว่านิ่งพอใช้ได้ สรุปว่าผลไม่ได้ขึ้นกับการเลือกเกณฑ์แบบเปราะบาง

## 4.11 ตรวจความสมเหตุสมผลด้วยฤดูกาล

ความคาดหวังที่เขียนไว้ก่อนดูผล คือหมอกควันระดับภูมิภาคของไทยมาช่วงฤดูหนาวถึงฤดูร้อน (ประมาณ พ.ย. ถึง เม.ย.)
ถ้าป้าย `regional` ของ PM2.5 กระจุกอยู่ช่วงนั้นจริง แปลว่ากฎที่ตั้งไว้จับสิ่งที่ควรจับได้

In [11]:
# ตรวจความสมเหตุสมผล ไม่มีคำตอบจริงให้เทียบ จึงดูว่าผลตรงกับความรู้เชิงกายภาพไหม
# ความคาดหวังที่เขียนไว้ก่อนดูผล: หมอกควันภูมิภาคควรกระจุกช่วงฤดูหนาวถึงฤดูร้อน
# ส่วน SO2 จากการเผาไหม้ในอุตสาหกรรมไม่ควรมีรูปแบบตามฤดูชัดเจน


def season(ts):
    m, d = ts.month, ts.day
    if (m == 2 and d >= 15) or m in (3, 4) or (m == 5 and d < 15):
        return 'summer'
    if (m == 5 and d >= 15) or m in (6, 7, 8, 9) or (m == 10 and d < 15):
        return 'rainy'
    return 'winter'


feat['season'] = feat['start_utc'].dt.tz_convert('Asia/Bangkok').apply(season)
for p in ('pm25', 'so2'):
    sub = feat[feat.pollutant == p]
    t = pd.crosstab(sub.season, sub.label, normalize='index').mul(100).round(1)
    t['จำนวน'] = pd.crosstab(sub.season, sub.label).sum(axis=1)
    print(f'=== {p.upper()} แยกตามฤดู (%) ===')
    print(t.to_string())
    print()

feat['wind_sectors_of_industry'] = feat['station_id'].map(lambda s: ' '.join(industrial_sectors[s]))
feat.to_csv('../data/processed/episodes_labeled.csv', index=False, encoding='utf-8-sig')
print(f'บันทึกเหตุการณ์ที่ติดป้ายแล้ว {len(feat):,} อัน ไว้ที่ data/processed/episodes_labeled.csv')

=== PM25 แยกตามฤดู (%) ===
label   local-consistent  regional  unclear-mixed  unclear-no-data  จำนวน
season                                                                   
rainy               27.3       0.0           66.7              6.1     33
summer              20.0      45.7           34.3              0.0     35
winter               5.1      67.8           23.7              3.4     59

=== SO2 แยกตามฤดู (%) ===
label   local-consistent  regional  unclear-mixed  unclear-no-data  จำนวน
season                                                                   
rainy               74.9       1.0            7.3             16.9    521
summer              58.4       2.5           11.7             27.3    315
winter              43.8       0.9           36.0             19.2    333

บันทึกเหตุการณ์ที่ติดป้ายแล้ว 1,796 อัน ไว้ที่ data/processed/episodes_labeled.csv


**อ่านผล (การตรวจที่สำคัญที่สุดของวันนี้):**

PM2.5 ที่ติดป้าย `regional` แบ่งตามฤดูได้ ฤดูหนาว 67.8% ฤดูร้อน 45.7% ฤดูฝน 0.0%
ตรงกับฤดูหมอกควันของไทยพอดี ทั้งที่กฎการติดป้ายไม่มีตัวแปรฤดูกาลอยู่ในสมการเลย

ส่วน SO2 ไม่มีรูปแบบตามฤดูแบบนั้น คือได้ `local-consistent` เป็นส่วนใหญ่ทุกฤดู (43.8% ถึง 74.9%)
ซึ่งสอดคล้องกับแหล่งกำเนิดที่ปล่อยต่อเนื่องตลอดปีแบบโรงงาน

นี่เป็นการตรวจแบบอิสระ เพราะข้อมูลฤดูไม่ได้ถูกใช้ในการติดป้ายเลย

## 4.12 Pollution rose และ CPF

CPF ตอบคำถามว่า "เวลาลมมาจากทิศนี้ โอกาสที่ค่าจะอยู่ใน 25% สูงสุดเป็นเท่าไร"

In [12]:
# CPF (Conditional Probability Function) = ความน่าจะเป็นที่ค่าจะอยู่ใน 25% สูงสุด เมื่อลมมาจากทิศนั้น
# ถ้า CPF สูงในทิศเดียวกับที่นิคมตั้งอยู่ แปลว่าค่าสูงสอดคล้องกับลมที่พัดผ่านนิคมมา
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = ['Leelawadee UI', 'Tahoma', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

STATION_LABEL = {7846: '7846 ใกล้นิคม 5.8 กม.', 225611: '225611 ใกล้นิคม 4.5 กม.'}
GRID, MUTED, INK = '#e1e0d9', '#898781', '#0b0b0b'
BAR, HILITE = '#1baf7a', '#eb6834'

fig, axes = plt.subplots(2, 2, figsize=(11, 11), subplot_kw={'projection': 'polar'})
fig.patch.set_facecolor('#fcfcfb')
summary = []

for ax, (sid, p) in zip(axes.ravel(), [(225611, 'so2'), (225611, 'pm25'), (7846, 'so2'), (7846, 'pm25')]):
    g = hourly[(hourly.station_id == sid) & hourly[p].notna() & (hourly.sector >= 0)]
    top_q = g[p].quantile(0.75)
    cpf = g.assign(hi=g[p] >= top_q).groupby('sector')['hi'].agg(['mean', 'size']).reindex(range(16), fill_value=0)
    theta = np.radians(np.arange(16) * 22.5)
    colors = [HILITE if LABELS[i] in industrial_sectors[sid] else BAR for i in range(16)]
    ax.bar(theta, cpf['mean'].values, width=np.radians(21), color=colors, edgecolor='white', linewidth=0.8)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_xticks(theta)
    ax.set_xticklabels(LABELS, fontsize=8, color=MUTED)
    ax.set_ylim(0, max(0.45, cpf['mean'].max() * 1.15))
    ax.set_yticks([0.2, 0.4])
    ax.set_yticklabels(['0.2', '0.4'], fontsize=7, color=MUTED)
    ax.grid(color=GRID, linewidth=0.8)
    ax.set_title(f"{STATION_LABEL[sid]}\nCPF ของ {p.upper()} (เกณฑ์ 25% สูงสุด = {top_q:g})",
                 fontsize=10.5, color=INK, pad=16)
    ind = [i for i in range(16) if LABELS[i] in industrial_sectors[sid]]
    summary.append({'station_id': sid, 'สาร': p,
                    'CPF เฉลี่ยทิศนิคม': round(cpf['mean'].iloc[ind].mean(), 3),
                    'CPF เฉลี่ยทิศอื่น': round(cpf['mean'].drop(index=ind).mean(), 3),
                    'ทิศที่ CPF สูงสุด': LABELS[int(cpf['mean'].idxmax())],
                    'ชั่วโมงที่ใช้': int(cpf['size'].sum())})

fig.text(0.5, 0.045, 'แท่งสีส้มคือทิศที่พื้นที่อุตสาหกรรมตั้งอยู่ (นิยามไว้ในหัวข้อ 4.4 ก่อนดูข้อมูลมลพิษ)',
         ha='center', fontsize=9, color=MUTED)
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig('../reports/day4_cpf_rose.png', dpi=150, facecolor=fig.get_facecolor())
plt.close(fig)
print('บันทึกกราฟแล้วที่ reports/day4_cpf_rose.png')
print()
print(pd.DataFrame(summary).to_string(index=False))

บันทึกกราฟแล้วที่ reports/day4_cpf_rose.png

 station_id  สาร  CPF เฉลี่ยทิศนิคม  CPF เฉลี่ยทิศอื่น ทิศที่ CPF สูงสุด  ชั่วโมงที่ใช้
     225611  so2              0.340              0.121                SW          21352
     225611 pm25              0.171              0.359                 N          29517
       7846  so2              0.332              0.225               SSE          20340
       7846 pm25              0.286              0.336               NNW          20825


**อ่านผล:** ดูรูปที่ `reports/day4_cpf_rose.png` ประกอบ

- SO2 ที่สถานี 225611 มี CPF ในทิศนิคม 0.340 เทียบกับทิศอื่น 0.121 คือสูงกว่าเกือบ 3 เท่า และทิศที่สูงสุดคือ SW ซึ่งอยู่ในทิศนิคมพอดี
- PM2.5 ที่สถานีเดียวกันกลับกันหมด คือทิศนิคม 0.171 แต่ทิศอื่น 0.359 และทิศที่สูงสุดคือ N ซึ่งไม่ใช่ทิศนิคม
- สถานี 7846 ให้ภาพเดียวกันแต่ชัดน้อยกว่า เพราะตัวสถานีถูกล้อมด้วยพื้นที่อุตสาหกรรมหลายทิศกว่า

ภาพนี้สนับสนุนผลการติดป้ายในหัวข้อ 4.9 แบบอิสระอีกชั้นหนึ่ง เพราะ CPF คำนวณจากข้อมูลรายชั่วโมงทั้งหมด
ไม่ได้ใช้ตารางเหตุการณ์หรือค่าสถานีควบคุมเลย

## 4.13 สุ่มตรวจด้วยตา 15 เหตุการณ์

แผนกำหนดให้สุ่มดู 10 ถึง 20 เหตุการณ์ด้วยตา แล้วรายงานว่าป้ายตรงกับกราฟหรือไม่
สุ่มแบบแบ่งชั้นตามป้าย (ป้ายละ 3 ถึง 4 อัน) ไม่ใช่สุ่มล้วน เพราะถ้าสุ่มล้วนจะได้แต่ `local-consistent` กับ `unclear`
แล้วจะไม่ได้ตรวจกฎ `regional` ซึ่งมีแค่ 89 อันเลย

In [13]:
# สุ่มเหตุการณ์มาตรวจด้วยตา แบ่งชั้นตามป้ายเพื่อให้ได้ครบทุกกฎ ไม่ใช่สุ่มล้วนซึ่งจะได้แต่ป้ายที่พบบ่อย
# ใช้ random_state ตายตัวเพื่อให้สุ่มซ้ำได้ผลเดิม
N_PER_LABEL = {'local-consistent': 4, 'regional': 4, 'unclear-mixed': 4, 'unclear-no-data': 3}
sample = pd.concat([feat[feat.label == lab].sample(n, random_state=42)
                    for lab, n in N_PER_LABEL.items()]).sort_values(['station_id', 'start_utc'])
sample = sample.reset_index(drop=True)
sample.insert(0, 'no', sample.index + 1)

cols = ['no', 'station_id', 'pollutant', 'start_utc', 'duration_hours', 'peak_value',
        'frac_wind_from_industry', 'frac_ctrl_flagged', 'median_increment', 'inc_threshold', 'label']
show = sample[cols].copy()
show['start_utc'] = show['start_utc'].dt.tz_convert('Asia/Bangkok').dt.strftime('%Y-%m-%d %H:%M')
show = show.rename(columns={'start_utc': 'เริ่ม (เวลาไทย)', 'duration_hours': 'ชม.',
                            'frac_wind_from_industry': 'ลมจากนิคม', 'frac_ctrl_flagged': 'ควบคุมผิดปกติ',
                            'median_increment': 'increment', 'inc_threshold': 'เกณฑ์'})
print(f'สุ่มมา {len(sample)} เหตุการณ์:')
print(show.to_string(index=False))

สุ่มมา 15 เหตุการณ์:
 no  station_id pollutant  เริ่ม (เวลาไทย)  ชม.  peak_value  ลมจากนิคม  ควบคุมผิดปกติ  increment  เกณฑ์            label
  1        7846        co 2018-10-10 16:00    2       0.960   0.000000       0.000000        NaN    NaN  unclear-no-data
  2        7846       no2 2018-10-12 13:00    1       0.027   1.000000       0.000000        NaN  0.006  unclear-no-data
  3        7846      pm10 2020-01-10 22:00    5      96.000   1.000000       1.000000     -2.000 -1.000         regional
  4        7846       no2 2020-05-04 13:00    1       0.036   1.000000       0.000000      0.034  0.006 local-consistent
  5        7846       so2 2020-05-09 14:00    2       0.054   1.000000       0.000000      0.038  0.002 local-consistent
  6        7846       so2 2020-06-26 15:00    1       0.009   0.000000       0.000000      0.009  0.002    unclear-mixed
  7        7846       so2 2020-07-01 01:00    1       0.014   0.000000       0.000000        NaN  0.002  unclear-no-data
  8        

วาดกราฟรายเหตุการณ์ไว้ดูด้วยตา แต่ละแผงแสดงค่าที่สถานีใกล้นิคมเทียบกับสถานีควบคุม
พร้อมจุดสี่เหลี่ยมด้านล่างบอกว่าชั่วโมงไหนลมพัดมาจากทิศนิคม

In [14]:
# วาดกราฟรายเหตุการณ์ไว้ตรวจด้วยตา แบ่งเป็น 3 รูป รูปละ 5 แผง
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams['font.family'] = ['Leelawadee UI', 'Tahoma', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
# matplotlib อ่านโซนเวลาจาก rcParams ไม่ได้อ่านจาก index ถ้าไม่ตั้งค่าแกน x จะเป็น UTC
# ทั้งที่หัวกราฟเป็นเวลาไทย ทำให้เวลาเหลื่อมกัน 7 ชั่วโมง
TZ = 'Asia/Bangkok'
plt.rcParams['timezone'] = TZ

NEAR_C, CTRL_C, SPAN_C = '#1baf7a', '#2a78d6', '#f2d9b8'
WIND_IN, WIND_OUT = '#eb6834', '#c3c2b7'
GRID, MUTED, INK = '#e1e0d9', '#898781', '#0b0b0b'

for fig_i in range(3):
    part = sample.iloc[fig_i * 5:(fig_i + 1) * 5]
    fig, axes = plt.subplots(5, 1, figsize=(11, 15))
    fig.patch.set_facecolor('#fcfcfb')
    for ax, ep in zip(axes, part.itertuples(index=False)):
        pad = pd.Timedelta(hours=max(12, ep.duration_hours))
        g = by_station[ep.station_id].loc[ep.start_utc - pad:ep.end_utc + pad]
        t = g.index.tz_convert('Asia/Bangkok')
        ax.set_facecolor('#fcfcfb')
        # เหตุการณ์ยาว 1 ชม. ถ้าระบายตามช่วงจริงจะบางจนมองไม่เห็น เลยขยายเป็นอย่างน้อย 1 ชม.เต็ม
        span_a = ep.start_utc.tz_convert(TZ) - pd.Timedelta(minutes=30)
        span_b = ep.end_utc.tz_convert(TZ) + pd.Timedelta(minutes=30)
        ax.axvspan(span_a, span_b, color=SPAN_C, alpha=0.55, lw=0)
        ax.plot(t, g[ep.pollutant], color=NEAR_C, lw=1.6, label=f'สถานี {ep.station_id} (ใกล้นิคม)')
        ax.plot(t, g[f'{ep.pollutant}_ctrl'], color=CTRL_C, lw=1.3, ls='--', label='สถานีควบคุม 717')

        # จุดสีด้านล่างบอกว่าชั่วโมงนั้นลมพัดมาจากทิศนิคมหรือไม่
        y0 = ax.get_ylim()[0]
        in_ind = g['sector'].map(lambda s: LABELS[s] in industrial_sectors[ep.station_id] if s >= 0 else None)
        for val, color in ((True, WIND_IN), (False, WIND_OUT)):
            m = in_ind == val
            ax.scatter(t[m], np.full(m.sum(), y0), s=9, color=color, marker='s', zorder=3)

        ax.set_title(f"#{ep.no} สถานี {ep.station_id} · {ep.pollutant.upper()} · "
                     f"{ep.start_utc.tz_convert('Asia/Bangkok'):%d %b %Y %H:%M} · {ep.duration_hours} ชม. "
                     f"· ป้าย: {ep.label}\n"
                     f"ลมจากนิคม {ep.frac_wind_from_industry:.0%} | ควบคุมผิดปกติ {ep.frac_ctrl_flagged:.0%} | "
                     f"increment {ep.median_increment:.4g} (เกณฑ์ {ep.inc_threshold:.4g})",
                     fontsize=9.5, color=INK, loc='left')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%H:%M', tz=TZ))
        ax.tick_params(labelsize=7.5, colors=MUTED)
        ax.grid(color=GRID, lw=0.7)
        for s in ax.spines.values():
            s.set_color(GRID)
        ax.legend(fontsize=7.5, loc='upper left', framealpha=0.9)
    fig.text(0.5, 0.005, 'แถบสีครีมคือช่วงเหตุการณ์ (ขยายข้างละ 30 นาทีให้มองเห็น) จุดสี่เหลี่ยมส้มด้านล่างคือชั่วโมงที่ลมพัดมาจากทิศนิคม เวลาทั้งหมดเป็นเวลาไทย',
             ha='center', fontsize=8.5, color=MUTED)
    fig.tight_layout(rect=[0, 0.012, 1, 1])
    fig.savefig(f'../reports/day4_manual_check_{fig_i + 1}.png', dpi=140, facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f'บันทึก reports/day4_manual_check_{fig_i + 1}.png (เหตุการณ์ #{part.no.min()} ถึง #{part.no.max()})')

บันทึก reports/day4_manual_check_1.png (เหตุการณ์ #1 ถึง #5)
บันทึก reports/day4_manual_check_2.png (เหตุการณ์ #6 ถึง #10)
บันทึก reports/day4_manual_check_3.png (เหตุการณ์ #11 ถึง #15)


### ผลการตรวจด้วยตา

ตารางนี้เป็นการตัดสินด้วยสายตาจากกราฟ ไม่ได้คำนวณจากโค้ด

| # | สถานี | สาร | ป้าย | สิ่งที่เห็นในกราฟ | คำตัดสิน |
|---|---|---|---|---|---|
| 1 | 7846 | CO | unclear-no-data | สถานีควบคุมไม่มีเส้นเลย คือไม่มีข้อมูล CO | ตรง |
| 2 | 7846 | NO2 | unclear-no-data | สถานีใกล้นิคมพุ่ง 0.027 แต่ชั่วโมงนั้นสถานีควบคุมขาดข้อมูลพอดี | ตรง แต่ระมัดระวังเกินไป |
| 3 | 7846 | PM10 | regional | ทั้งสองสถานีสูง 90 ถึง 110 พร้อมกัน ควบคุมสูงกว่าด้วยซ้ำ | ตรง ชัดเจน |
| 4 | 7846 | NO2 | local-consistent | ใกล้นิคมพุ่ง 0.036 ควบคุมนิ่งที่ 0.003 ลมจากนิคม 100% | ตรง ชัดเจน |
| 5 | 7846 | SO2 | local-consistent | ใกล้นิคมพุ่ง 0.054 ควบคุมแบนราบที่ 0 ลมจากนิคม 100% | ตรง ชัดเจน |
| 6 | 7846 | SO2 | unclear-mixed | ใกล้นิคมสูงกว่าควบคุมชัด แต่ลมไม่ได้มาจากทิศนิคม | ตรง |
| 7 | 7846 | SO2 | unclear-no-data | สถานีควบคุมขาดข้อมูลช่วงเหตุการณ์พอดี | ตรง |
| 8 | 7846 | PM2.5 | regional | ควบคุมสูงกว่าใกล้นิคมตลอด ทั้งคู่อยู่ระดับ 40 ถึง 67 | ตรง ชัดเจน |
| 9 | 7846 | PM2.5 | regional | เหตุการณ์ยาว 63 ชม. ทั้งสองสถานีขึ้นลงพร้อมกัน | ตรง ชัดเจน |
| 10 | 225611 | NO2 | unclear-mixed | ใกล้นิคมสูงกว่าควบคุม แต่ลมไม่ได้มาจากทิศนิคม | ตรง |
| 11 | 225611 | SO2 | local-consistent | ใกล้นิคมขึ้น ควบคุมนิ่ง ลมจากนิคม 100% | ตรง |
| 12 | 225611 | O3 | unclear-mixed | ทั้งสองสถานีขึ้นลงพร้อมกันเป็นรูประฆัง ดูเหมือน regional มากกว่า | ก้ำกึ่ง |
| 13 | 225611 | SO2 | regional | ใกล้นิคมพุ่ง 0.048 **ก่อน** แล้วควบคุมค่อยขึ้นไป 0.013 ทีหลัง สูงกว่ากัน 3.7 เท่า | น่าสงสัย |
| 14 | 225611 | SO2 | local-consistent | ใกล้นิคมพุ่ง 0.037 ควบคุมนิ่งที่ 0.001 ลมจากนิคม 100% | ตรง ชัดเจน |
| 15 | 225611 | O3 | unclear-mixed | ทั้งสองสถานีขึ้นลงพร้อมกัน คล้ายกรณี #12 | ก้ำกึ่ง |

**สรุป: ตรง 12 อัน, ก้ำกึ่ง 2 อัน, น่าสงสัย 1 อัน จาก 15 อัน**

**จุดอ่อนที่เจอจากการตรวจด้วยตา** ทั้ง 3 กรณีที่ไม่ผ่านชี้ไปที่รากเดียวกัน
คือกฎ `regional` ตัดสินจาก "สถานีควบคุมติด flag ผิดปกติหรือไม่" ซึ่งเป็นค่าจริงเท็จ ไม่ได้เทียบขนาดของค่าทั้งสองสถานี

- กรณี #13 สถานีควบคุมบังเอิญติด flag ด้วย เลยได้ป้าย `regional` ทั้งที่ค่าที่สถานีใกล้นิคมสูงกว่า 3.7 เท่าและขึ้นก่อน
  ซึ่งรูปแบบแบบนี้สอดคล้องกับกลุ่มควันจากแหล่งในพื้นที่ที่ลอยไปถึงสถานีควบคุมทีหลังมากกว่า
- กรณี #12 กับ #15 ทั้งสองสถานีขึ้นลงพร้อมกันชัดเจน แต่สถานีควบคุมไม่ได้ผิดปกติถึงเกณฑ์ของตัวเอง เลยไม่ได้ `regional`

**ข้อเสนอให้แก้ในรอบถัดไป** เพิ่มเงื่อนไขเชิงขนาดเข้าไปในกฎ `regional` เช่น ต้องให้ค่าที่สถานีควบคุม
ไม่ต่ำกว่าค่าที่สถานีใกล้นิคมเกินครึ่ง หรือใช้สหสัมพันธ์ของอนุกรมเวลาสองสถานีในช่วงเหตุการณ์แทนการใช้ flag อย่างเดียว

## สรุป Day 4

ติดป้ายเหตุการณ์ได้ 1,796 อัน แบ่งเป็น `local-consistent` 954 อัน, `regional` 89 อัน และ `unclear` 753 อัน
ผลบันทึกไว้ที่ `data/processed/episodes_labeled.csv`

**ข้อจำกัดที่ต้องเขียนลง README**
1. สถานีควบคุม 717 ไม่ใช่สถานีพื้นหลังที่สะอาด มีพื้นที่อุตสาหกรรมในรัศมี 10 กม. ถึง 40.9 ตร.กม. ซึ่งมากกว่าสถานีใกล้นิคมเสียอีก
   แก้ปัญหาด้วยการตัดชั่วโมงที่ลมพัดจากนิคมเข้าสถานีควบคุมออก แต่ก็ยังไม่ใช่ทางออกที่สมบูรณ์
2. ป้าย `unclear` มีถึง 42% ซึ่งส่วนใหญ่มาจากข้อมูลไม่ครบ ไม่ใช่จากกฎที่ตัดสินไม่ได้
3. การถ่วงน้ำหนักด้วย 1/d² เป็นการประมาณการเจือจางแบบหยาบ ไม่ใช่แบบจำลองการกระจายตัวของมลพิษจริง
4. ลมจาก Open-Meteo เป็นข้อมูลจากโมเดล grid 9 กม. ซึ่งอาจจับลมบกลมทะเลใกล้ชายฝั่งได้ไม่ดี
5. ป้ายกลุ่มไม่มีคำตอบจริงให้เทียบ จึงรายงานการทดสอบความไว การตรวจด้วยฤดูกาล และการสุ่มตรวจด้วยตา แทนคำว่า "ความแม่นยำ"
6. CO และ PM10 ที่สถานี 225611 ยังติดป้ายไม่ได้เพราะข้อมูลคู่สถานีไม่พอ
7. กฎ `regional` ใช้ flag ของสถานีควบคุมอย่างเดียว ไม่ได้เทียบขนาดของค่า ทำให้พลาดได้ทั้งสองทางตามที่เจอในหัวข้อ 4.13

**ผลการตรวจสอบ 3 ชั้น (ไม่มีคำตอบจริงให้เทียบ จึงต้องตรวจทางอ้อม)**
1. ความไวต่อเกณฑ์ ขยับเกณฑ์แล้วป้ายเปลี่ยนไม่เกิน 2.9% (เกณฑ์ increment ที่ p90 เปลี่ยน 6.0%)
2. ฤดูกาล PM2.5 ที่ติด `regional` อยู่ในฤดูหนาว 67.8% ฤดูร้อน 45.7% ฤดูฝน 0.0% ตรงกับฤดูหมอกควัน ทั้งที่กฎไม่มีตัวแปรฤดู
3. สุ่มตรวจด้วยตา 15 อัน ตรง 12 อัน ก้ำกึ่ง 2 อัน น่าสงสัย 1 อัน

**สิ่งที่ยังควรทำต่อถ้ามีเวลา**
- ดึงสถานี 225690 มาเป็นสถานีควบคุมที่สอง แล้วเทียบว่าป้ายเปลี่ยนไปมากไหม
- ปรับกฎ `regional` ให้เทียบขนาดของค่าสองสถานี ไม่ใช่ดูแค่ flag